# Capítulo 3 — Representação do problema: reenquadrar, cascata e rebalancear

## Missão
Capítulo 3, padrões 5 a 10. Muitas vezes o que muda o resultado é **como se formula** o problema, não o algoritmo. Você vai:
- **reenquadrar** a regressão do tempo como previsão de **intervalos** (quantis), porque o cliente prefere "30 a 40 min" a "34,2 min";
- prever **várias etiquetas** de um comentário (*multilabel*);
- **combinar modelos** (*ensembles*);
- montar uma **cascata** para os pedidos grandes;
- criar uma **classe neutra** para a nota 3;
- **rebalancear** a previsão de cancelamento, que é raro (1%).

**No fim deste caderno você terá:** o módulo `meu_mlops/problema.py` (intervalos por quantis e cobertura, cascata e avaliação de classes raras) e as métricas de cada formulação.

**Leitura essencial (opcional, livro em inglês):**
- Design Pattern 5: Reframing · PDF p. 111
- Design Pattern 6: Multilabel · PDF p. 120
- Design Pattern 7: Ensembles · PDF p. 127
- Design Pattern 8: Cascade · PDF p. 134
- Design Pattern 9: Neutral Class · PDF p. 141
- Design Pattern 10: Rebalancing · PDF p. 145

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap03_representacao_do_problema_reenquadrar_cascata.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_mlops/
os.makedirs("../meu_mlops", exist_ok=True)
open("../meu_mlops/__init__.py", "a").close()        # meu_mlops/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_mlops")

## Ideia em 30 segundos
- **Reframing:** regressão ↔ classificação, ou prever a distribuição (quantis) em vez de um número.
- **Cascade:** um primeiro modelo decide **qual** modelo especialista usar, útil quando há subgrupos com comportamento diferente.
- **Neutral class:** quando os especialistas discordam ou o caso é ambíguo, uma terceira classe evita forçar a escolha.
- **Rebalancing:** com classes raras, a acurácia mente. Use pesos, subamostragem, ou trate o problema como detecção de anomalia.

### 3.1 **Funções do capítulo.** Crie `problema.py` com:
- `treinar_quantis(X, y, quantis=(.1, .5, .9))`: um `GradientBoostingRegressor(loss="quantile", alpha=q)` por quantil;
- `cobertura(y, baixo, alto)`: a fração dentro do intervalo;
- `Cascata(roteador, especialistas)`: um `fit(X, y, grupo)` que treina o roteador para prever o grupo e um especialista por grupo, e um `predict`;
- `relatorio_raro(y, prob, limiar)`: precisão, revocação e alertas.

_A célula grava `meu_mlops/problema.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_mlops/problema.py


### 3.2 **Reenquadrar: um intervalo.** Treine os quantis 10%, 50% e 90% do tempo de entrega e mostre, para 5 pedidos do teste, a frase que o app exibiria ("entre X e Y min"). Meça a **cobertura** do intervalo 10-90% (deve ficar perto de 80%) e a largura média.

### 3.3 **Multilabel.** Cada comentário pode ter várias etiquetas: `demora` ("demorou", "frio"), `erro_pedido` ("faltou", "errou", "derramado") e `elogio` ("rápido", "ótima", "perfeito", "caprichada"). Crie as etiquetas por regra, treine um `OneVsRestClassifier(LogisticRegression)` sobre TF-IDF **sem** usar as palavras da regra como entrada direta (use só os números do pedido) e veja quais etiquetas são previsíveis pelos números.

### 3.4 **Cascata.** Pedidos com 5 itens ou mais têm outra dinâmica (o preparo domina). Monte uma `Cascata` com um roteador (`DecisionTreeClassifier` que prevê "grande") e especialistas `Ridge`, e compare o MAE com um único `Ridge` e com um único `GradientBoostingRegressor`.

### 3.5 **Classe neutra.** Para prever o "humor" do cliente pela nota, compare: (a) duas classes, positivo (≥ 4) × negativo (≤ 3); (b) três classes, com a nota 3 como **neutra**. Nos dois casos, use o tempo de entrega e a distância. Na formulação (b), quantos clientes "negativos" de (a) eram, na verdade, neutros? Mostre também a precisão da classe negativa em cada formulação.

### 3.6 **Rebalancear o cancelamento (1%).** Compare três estratégias de `LogisticRegression` para prever `cancelado`:
- (a) sem nada;
- (b) com `class_weight="balanced"`;
- (c) com subamostragem dos não cancelados para 5×.

Use as numéricas + o tempo de entrega. Mostre a AUC e o `relatorio_raro` com o limiar que dá 50% de revocação.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
